In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed2026_a_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed123_d_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed3407_b_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed2

## Setup

In [2]:
import gc, random, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
 
SEEDS   = [999, 7777, 31415, 8888, 2024, 1111, 5050, 6666]
N_FOLDS = 5
TARGET  = 'Irrigation_Need'

## Load data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')
orig  = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv')
 
train = pd.concat([train, orig], ignore_index=True)
 
target2idx = {v: i for i, v in enumerate(train[TARGET].unique())}
idx2target = {v: k for k, v in target2idx.items()}
train[TARGET] = train[TARGET].map(target2idx)
y = train[TARGET].values
train.drop(columns=['id'], inplace=True)
test.drop(columns=['id'], inplace=True)

## Feature Engineering 

In [4]:
CATS = [c for c in test.columns if test[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]
M    = train[NUMS].max()
 
def FE(df):
    out = df.copy()
    for c in NUMS:
        for k in range(-4, 4):
            out[f"{c}_digit{k}"] = (out[c] // (10**k) % 10).astype('int8')
        out[c] = out[c].round(3 if M[c]<10 else (2 if M[c]<100 else 1))
    return out
 
train_fe = FE(train.drop(TARGET, axis=1))
test_fe  = FE(test)
 
DROP = [c for c in test_fe.columns if test_fe[c].nunique() == 1]
train_fe.drop(columns=DROP, inplace=True)
test_fe.drop(columns=DROP, inplace=True)
 
CATEGORY = CATS + [c for c in test_fe.columns if 'digit' in c]
for c in CATEGORY:
    freq    = train_fe[c].value_counts()
    mapping = {v: i for i, (v, cnt) in enumerate(freq[freq >= 5].items())}
    defval  = len(mapping)
    train_fe[c] = train_fe[c].map(lambda x: mapping.get(x, defval))
    test_fe[c]  = test_fe[c].map(lambda x: mapping.get(x, defval))
 
FEATURES = CATEGORY + [c for c in NUMS if c in test_fe.columns]

## OrderedTE

In [5]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a
    def fit(self, train_df, category_cols, target_col):
        self.category_cols = category_cols
        self.classes_      = sorted(train_df[target_col].unique())
        self.global_prior_ = train_df[target_col].value_counts(normalize=True).sort_index().values
        self._stats = {}
        for c in category_cols:
            stats = {}
            for k, cls in enumerate(self.classes_):
                y_bin = (train_df[target_col] == cls).astype(int)
                df    = train_df[[c]].copy()
                df['y'] = y_bin.values; df['cnt'] = 1
                df['cum_cnt'] = df.groupby(c)['cnt'].cumsum() - df['cnt']
                df['cum_sum'] = df.groupby(c)['y'].cumsum() - df['y']
                pr = self.a * self.global_prior_[k]
                te = (df['cum_sum'] + pr) / (df['cum_cnt'] + self.a)
                te[df['cum_cnt'] == -1] = self.global_prior_[k]
                self.__dict__[f'_te_{c}_{cls}'] = te.values
                agg = df.groupby(c)['y'].agg(['count','sum']).reset_index()
                agg.columns = [c, f'cnt_{cls}', f'sm_{cls}']
                agg[f'pr_{cls}'] = self.global_prior_[k]
                stats[cls] = agg
            self._stats[c] = stats
        return self
    def transform_train(self, train_df, category_cols):
        out = train_df.copy()
        for c in category_cols:
            for k, cls in enumerate(self.classes_):
                out[f'{c}_TE_cls{cls}'] = self.__dict__[f'_te_{c}_{cls}']
        return out
    def transform(self, df, category_cols):
        out = df.copy()
        for c in category_cols:
            for k, cls in enumerate(self.classes_):
                agg = self._stats[c][cls]; pr = float(self.global_prior_[k])
                out = out.merge(agg, on=c, how='left')
                te  = (out[f'sm_{cls}'].fillna(0) + self.a*pr) / (out[f'cnt_{cls}'].fillna(0) + self.a)
                out[f'{c}_TE_cls{cls}'] = te.fillna(pr).astype(np.float32)
                out.drop(columns=[f'cnt_{cls}',f'sm_{cls}',f'pr_{cls}'], inplace=True, errors='ignore')
        return out

## Sample weights

In [6]:
unique, counts = np.unique(y, return_counts=True)
weights_dict   = {cls: len(y)/len(unique)/cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([weights_dict[lbl] for lbl in y])

## XGB params - max_depth=4

In [7]:
XGB_PARAMS = dict(
    max_depth=4, learning_rate=0.021, n_estimators=3038,
    min_child_weight=3, subsample=0.734, colsample_bytree=0.505,
    colsample_bylevel=0.704, colsample_bynode=0.539,
    reg_alpha=5.7e-5, reg_lambda=7.843, gamma=0.018,
    objective='multi:softprob', num_class=3,
    device='cuda', tree_method='hist', eval_metric='mlogloss', n_jobs=-1,
)
 
best_cv, best_lb_estimate, best_file = 0, 0, ''
 
for seed in SEEDS:
    print(f"\n{'='*50}  SEED={seed}")
    np.random.seed(seed); random.seed(seed)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros((len(train_fe), 3))
    tst = np.zeros((len(test_fe), 3))
 
    for fold, (tri, vai) in enumerate(kf.split(train_fe)):
        Xtr=train_fe.iloc[tri].copy(); Xva=train_fe.iloc[vai].copy(); Xte=test_fe.copy()
        ytr=y[tri]; yva=y[vai]; wtr=sample_weights[tri]
        te=OrderedTE(a=1)
        tr_full=pd.concat([Xtr, pd.Series(ytr,name=TARGET,index=Xtr.index)],axis=1)
        tr_full['_w']=wtr
        frames=[]
        for i in range(4):
            sh=tr_full.sample(frac=1,random_state=seed+i).reset_index(drop=True)
            te.fit(sh, FEATURES, TARGET); enc=te.transform_train(sh, FEATURES); frames.append(enc)
        aug=pd.concat(frames,ignore_index=True)
        ytr_aug=aug[TARGET].values; wtr_aug=aug['_w'].values
        aug.drop(columns=[TARGET,'_w']+[c for c in CATEGORY if c in aug.columns],inplace=True)
        Xva_enc=te.transform(Xva,FEATURES); Xte_enc=te.transform(Xte,FEATURES)
        Xva_enc.drop(columns=[c for c in CATEGORY if c in Xva_enc.columns],inplace=True)
        Xte_enc.drop(columns=[c for c in CATEGORY if c in Xte_enc.columns],inplace=True)
        fc=[c for c in aug.columns if c in Xva_enc.columns]
        params={**XGB_PARAMS,'random_state':seed}
        m=XGBClassifier(**params)
        m.fit(aug[fc],ytr_aug,sample_weight=wtr_aug)
        oof[vai]=m.predict_proba(Xva_enc[fc])
        tst+=m.predict_proba(Xte_enc[fc])/N_FOLDS
        s=balanced_accuracy_score(yva,np.argmax(oof[vai],axis=1))
        print(f"  Fold {fold+1}/{N_FOLDS} bACC={s:.5f}")
        del m; gc.collect()
 
    cv=balanced_accuracy_score(y,np.argmax(oof,axis=1))
    print(f"SEED={seed} CV={cv:.5f}")
 
    def obj(trial):
        cw=np.array([trial.suggest_float(f'cw{i}',0.5,3.0) for i in range(3)])
        adj=oof*cw; adj=adj/adj.sum(axis=1,keepdims=True)
        return balanced_accuracy_score(y,np.argmax(adj,axis=1))
    study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(obj,n_trials=200,show_progress_bar=False)
    best_cw=np.array([study.best_params[f'cw{i}'] for i in range(3)])
    print(f"Optuna={study.best_value:.5f}  cw={best_cw.round(3)}")
 
    adj_test=tst*best_cw; adj_test=adj_test/adj_test.sum(axis=1,keepdims=True)
    preds=np.argmax(adj_test,axis=1)
    fname=f'submission_d4_s{seed}.csv'
    out=sub.copy(); out['Irrigation_Need']=[idx2target[p] for p in preds]
    out.to_csv(fname,index=False)
    print(f"Saved {fname}")
 
    if study.best_value > best_lb_estimate:
        best_lb_estimate = study.best_value
        best_file = fname
 
print(f"\nBest Optuna CV: {best_lb_estimate:.5f} → {best_file}")
print("Submit the highest Optuna CV file first!")


==================================================  SEED=999
  Fold 1/5 bACC=0.98047
  Fold 2/5 bACC=0.97888
  Fold 3/5 bACC=0.97813
  Fold 4/5 bACC=0.97885
  Fold 5/5 bACC=0.98016
SEED=999 CV=0.97929
Optuna=0.98030  cw=[1.203 1.389 2.51 ]
Saved submission_d4_s999.csv

==================================================  SEED=7777
  Fold 1/5 bACC=0.98042
  Fold 2/5 bACC=0.97746
  Fold 3/5 bACC=0.97936
  Fold 4/5 bACC=0.97887
  Fold 5/5 bACC=0.97949
SEED=7777 CV=0.97913
Optuna=0.98006  cw=[1.777 1.755 2.844]
Saved submission_d4_s7777.csv

==================================================  SEED=31415
  Fold 1/5 bACC=0.97837
  Fold 2/5 bACC=0.97821
  Fold 3/5 bACC=0.98010
  Fold 4/5 bACC=0.98019
  Fold 5/5 bACC=0.97990
SEED=31415 CV=0.97935
Optuna=0.98047  cw=[1.418 1.64  2.805]
Saved submission_d4_s31415.csv

==================================================  SEED=8888
  Fold 1/5 bACC=0.97908
  Fold 2/5 bACC=0.98046
  Fold 3/5 bACC=0.97847
  Fold 4/5 bACC=0.97964
  Fold 5/5 bACC=0.9791